[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/PythonForDataAnalysis/blob/master/Numpy%E5%BB%BA%E7%AB%8B%E8%88%87%E8%A8%88%E7%AE%97/%E5%AD%B8%E7%94%9F%E7%B4%A0%E6%9D%90/%E7%AF%84%E4%BE%8B_%E4%BD%A0%E5%AE%B6%E9%99%84%E8%BF%91%E7%9A%84%E7%A9%BA%E6%B0%A3%E5%A5%BD%E5%97%8E_365%E5%A4%A9AQI%E7%9F%A9%E9%99%A3.ipynb)

# 🌫️ 你家附近的空氣好嗎？365 天 × 14 個測站的 AQI 矩陣

> 🎯 **情境**：早上出門前，你會看手機上的空氣品質嗎？環境部全台有 80 多個測站，每天公布空氣品質指標 AQI（0～50 良好、51～100 普通、101 以上對敏感族群不健康）。你拿到其中 14 個測站、整整一年的資料，一共 5,000 多個數字。今天要用 **NumPy 一次算完整張表**，回答：**哪裡的空氣最差？什麼季節最糟？**

| 素材檔案 | 內容 |
| --- | --- |
| `每日AQI_14測站.csv` | 近一年（2025-09-30～2026-10-23）14 個測站的每日空氣品質指標 AQI；整理自環境部「日空氣品質指標」，**空白代表那天測站沒有資料** |

| 📡 資料來源（政府資料開放平台） | 提供機關 | 下載日期 |
| --- | --- | --- |
| [日空氣品質指標(AQI)](https://data.gov.tw/dataset/40507) | 環境部 | 2026-10-24 |

> ✅ 這些都是**真實的政府開放資料**，依「政府資料開放授權條款－第1版」使用。課堂上統一使用上表日期的快照，所以每個人跑出來的數字都一樣。

## 🗺️ 今日任務地圖

| 關卡 | 要回答的問題 |
| --- | --- |
| ⬜ 任務 1 | 把 CSV 讀成一個 NumPy 矩陣 |
| ⬜ 任務 2 | 每個測站的年平均——遇到 nan 怎麼辦？ |
| ⬜ 任務 3 | 全台平均最差的是哪一天？ |
| ⬜ 任務 4 | 一年之中，有幾天「對敏感族群不健康」？ |
| ⬜ 任務 5 | 替每一天貼上標籤——像 Excel 的 IF |
| ⬜ 任務 6 | 什麼季節空氣最差？ |

**怎麼用這本 Notebook**：從上到下按 `Shift + Enter` 一格一格執行。每個任務都會先講「你想知道什麼」，再給程式，最後用 💡 **看懂結果** 告訴你這個數字代表什麼。

In [1]:
# 📦 準備素材：在 Colab 開啟時，自動下載本章素材壓縮檔並解壓縮
#    （若是在自己電腦解壓縮後開啟，檔案已經在旁邊，會直接使用）
import os, zipfile, urllib.request

FILES = [
    '每日AQI_14測站.csv',
]
ZIP_URL = 'https://github.com/roberthsu2003/PythonForDataAnalysis/raw/master/Numpy%E5%BB%BA%E7%AB%8B%E8%88%87%E8%A8%88%E7%AE%97/Numpy%E5%BB%BA%E7%AB%8B%E8%88%87%E8%A8%88%E7%AE%97_%E5%AD%B8%E7%94%9F%E7%B4%A0%E6%9D%90.zip'

if not all(os.path.exists(f) for f in FILES):
    urllib.request.urlretrieve(ZIP_URL, 'materials.zip')            # 下載本章素材壓縮檔
    with zipfile.ZipFile('materials.zip') as z:
        for name in z.namelist():
            inner = name.split('/', 1)[1] if '/' in name else name   # 去掉壓縮檔最外層的資料夾名稱
            if inner and not name.endswith('/'):
                os.makedirs(os.path.dirname(inner) or '.', exist_ok=True)
                with z.open(name) as src, open(inner, 'wb') as dst:
                    dst.write(src.read())

print('✅ 素材準備完成：', [f for f in FILES if os.path.exists(f)])

✅ 素材準備完成： ['每日AQI_14測站.csv']


## 任務 1：把 CSV 讀成一個 NumPy 矩陣

**白話說明**：NumPy 的陣列就像一張「全部都是數字」的表格。`np.genfromtxt` 讀數字，遇到空白會自動變成 `nan`（Not a Number，代表「沒有資料」）。日期是文字，另外用 `np.loadtxt(..., dtype=str)` 讀。

In [2]:
import numpy as np

with open('每日AQI_14測站.csv', encoding='utf-8') as f:
    sites = f.readline().strip().split(',')[1:]          # 第一行是標題：取出 14 個測站名稱

dates = np.loadtxt('每日AQI_14測站.csv', delimiter=',', skiprows=1, usecols=0, dtype=str)   # 第 0 欄：日期
aqi = np.genfromtxt('每日AQI_14測站.csv', delimiter=',', skip_header=1, usecols=range(1, 15))  # 第 1～14 欄：AQI

print('測站：', sites)
print('矩陣形狀：', aqi.shape, '→', aqi.shape[0], '天 ×', aqi.shape[1], '個測站')
print('日期範圍：', dates[0], '～', dates[-1])
print('前 3 天：\n', aqi[:3])

測站： ['基隆', '林森', '中山', '板橋', '桃園', '新竹', '忠明', '斗六', '臺南', '小港', '屏東', '花蓮', '臺東', '金門']
矩陣形狀： (365, 14) → 365 天 × 14 個測站
日期範圍： 2025-09-30 ～ 2026-10-23
前 3 天：
 [[40. nan 57. 87. 55. 50. 55. 62. 44. 52. 49. 30. 28. 77.]
 [51. nan 54. 93. 67. 50. 64. 69. 44. 61. 58. 25. 20. 64.]
 [29. nan 45. 41. 61. 54. 58. 58. 46. 61. 46. 31. 19. 77.]]


## 任務 2：每個測站的年平均——遇到 nan 怎麼辦？

**你想知道**：一整年下來，哪一個測站的空氣最差？

**白話說明**：`axis=0` 是「**直著算**」（每一欄、每個測站一個答案）；`axis=1` 是「**橫著算**」（每一列、每一天一個答案）。

In [3]:
site_mean = aqi.mean(axis=0)                             # 直著算：每個測站的平均
print(dict(zip(sites, site_mean.round(1).tolist())))

{'基隆': 50.8, '林森': nan, '中山': 57.4, '板橋': 57.6, '桃園': 55.4, '新竹': 53.8, '忠明': 59.8, '斗六': 75.0, '臺南': 65.9, '小港': 73.7, '屏東': 70.7, '花蓮': 41.3, '臺東': 42.2, '金門': 73.6}


> 😱 **怪了！** 「林森」測站的平均是 `nan`？因為它有幾十天沒資料，**只要有一個 `nan`，整個平均就變成 `nan`**。
>
> 解法：用 `np.nanmean`，它會**跳過 nan**，只用有資料的天數算平均。

In [4]:
site_mean = np.nanmean(aqi, axis=0)                      # 跳過 nan 再算平均
missing = np.isnan(aqi).sum(axis=0)                      # 每個測站有幾天沒資料

order = np.argsort(site_mean)[::-1]                      # 由差到好排序（argsort 由小到大，[::-1] 反轉）
for i in order:
    note = f'（{missing[i]} 天沒資料）' if missing[i] else ''
    print(f'{sites[i]:　<4} 平均 AQI {site_mean[i]:5.1f} ' + '█' * int(site_mean[i] / 4) + note)

斗六　　 平均 AQI  75.0 ██████████████████
小港　　 平均 AQI  73.7 ██████████████████
金門　　 平均 AQI  73.6 ██████████████████
屏東　　 平均 AQI  70.7 █████████████████
臺南　　 平均 AQI  65.9 ████████████████
林森　　 平均 AQI  64.2 ████████████████（65 天沒資料）
忠明　　 平均 AQI  59.8 ██████████████
板橋　　 平均 AQI  57.6 ██████████████
中山　　 平均 AQI  57.4 ██████████████
桃園　　 平均 AQI  55.4 █████████████
新竹　　 平均 AQI  53.8 █████████████
基隆　　 平均 AQI  50.8 ████████████
臺東　　 平均 AQI  42.2 ██████████
花蓮　　 平均 AQI  41.3 ██████████


> 💡 **看懂結果**：空氣最差的是**斗六、小港、金門、屏東**（平均超過 70），最好的是**花蓮、臺東**（只有 40 出頭）——差了快一倍！
> 斗六、小港附近有工業區與重工業；金門離中國沿海很近，常受境外污染影響；花東有中央山脈擋住西半部的污染。**你住的地方，空氣是哪一種？**
>
> 🤖 **換你問 AI**：「我有一個 365 × 14 的 NumPy 陣列，裡面有 nan。請算出每一欄的平均（跳過 nan），並由大到小列出欄位名稱。」

## 任務 3：全台平均最差的是哪一天？

**白話說明**：換成 `axis=1`——「橫著算」，每一天得到一個 14 個測站的平均。再用 `argmax` / `argmin` 找出最大、最小值在**第幾個位置**，拿這個位置去 `dates` 查日期。

In [5]:
day_mean = np.nanmean(aqi, axis=1)                       # 橫著算：每天 14 個測站的平均（365 個數字）

worst, best = day_mean.argmax(), day_mean.argmin()       # 最大、最小值的位置
print(f'😷 最差的一天：{dates[worst]}，平均 AQI {day_mean[worst]:.0f}')
print(f'😊 最好的一天：{dates[best]}，平均 AQI {day_mean[best]:.0f}')
print('那一天各測站：', dict(zip(sites, aqi[worst].tolist())))

😷 最差的一天：2026-10-13，平均 AQI 131
😊 最好的一天：2026-07-05，平均 AQI 28
那一天各測站： {'基隆': 140.0, '林森': 93.0, '中山': 112.0, '板橋': 132.0, '桃園': 140.0, '新竹': 150.0, '忠明': 132.0, '斗六': 146.0, '臺南': 159.0, '小港': 132.0, '屏東': 151.0, '花蓮': 112.0, '臺東': 77.0, '金門': 159.0}


## 任務 4：一年之中，有幾天「對敏感族群不健康」？

**你想知道**：有氣喘的同學，一年有幾天出門要特別小心？

**白話說明**：`aqi > 100` 會對**每一格**判斷，得到一張 True / False 的表。True 會被當成 1，所以 `.sum(axis=0)` 就是每個測站「超過 100 的天數」。（`nan > 100` 會是 False，不會被算進去。）

In [6]:
unhealthy = aqi > 100                                    # 365 × 14 的 True / False
bad_days = unhealthy.sum(axis=0)                         # 直著數：每個測站幾天

for i in np.argsort(bad_days)[::-1]:
    print(f'{sites[i]:　<4} {bad_days[i]:3d} 天 ' + '▇' * int(bad_days[i] / 2))

金門　　  69 天 ▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
斗六　　  69 天 ▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
屏東　　  50 天 ▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
小港　　  38 天 ▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇
臺南　　  28 天 ▇▇▇▇▇▇▇▇▇▇▇▇▇▇
林森　　  21 天 ▇▇▇▇▇▇▇▇▇▇
忠明　　  17 天 ▇▇▇▇▇▇▇▇
板橋　　  14 天 ▇▇▇▇▇▇▇
桃園　　  13 天 ▇▇▇▇▇▇
新竹　　   9 天 ▇▇▇▇
中山　　   7 天 ▇▇▇
基隆　　   6 天 ▇▇▇
臺東　　   2 天 ▇
花蓮　　   2 天 ▇


> 💡 **看懂結果**：斗六和金門一年有將近 **70 天**（大約每 5 天就有 1 天）空氣對敏感族群不健康，花蓮、臺東只有 2 天。**同一個台灣，呼吸到的空氣差很多。**

## 任務 5：替每一天貼上標籤——像 Excel 的 IF

**白話說明**：`np.where(條件, 符合時, 不符合時)` 就像 Excel 的 IF；兩層 `np.where` 就能分三級。`np.unique(..., return_counts=True)` 數每一種標籤出現幾次。

In [7]:
zhongshan = aqi[:, sites.index('中山')]                   # 取出台北中山站那一欄
label = np.where(zhongshan <= 50, '良好', np.where(zhongshan <= 100, '普通', '不健康'))

kinds, counts = np.unique(label, return_counts=True)
print('台北中山站一年的空氣：', dict(zip(kinds, counts.tolist())))
print('最近 7 天：', list(zip(dates[-7:], label[-7:])))

台北中山站一年的空氣： {np.str_('不健康'): 7, np.str_('普通'): 264, np.str_('良好'): 94}
最近 7 天： [(np.str_('2026-10-17'), np.str_('良好')), (np.str_('2026-10-18'), np.str_('良好')), (np.str_('2026-10-19'), np.str_('良好')), (np.str_('2026-10-20'), np.str_('良好')), (np.str_('2026-10-21'), np.str_('良好')), (np.str_('2026-10-22'), np.str_('良好')), (np.str_('2026-10-23'), np.str_('良好'))]


## 任務 6：什麼季節空氣最差？

**白話說明**：日期是 `'2026-01-15'` 這樣的文字，取前 7 個字 `'2026-01'` 就是月份。用「布林遮罩」挑出某個月的那幾列，再算平均。

In [8]:
months = np.array([d[5:7] for d in dates])               # '2026-01-15' → '01'

print('月份  全台平均 AQI')
for m in sorted(set(months)):
    mask = months == m                                   # 這個月的那幾天是 True
    v = np.nanmean(aqi[mask])                            # 這幾天 × 14 個測站，全部一起平均
    print(f' {m} 月   {v:5.1f} ' + '█' * int(v / 2))

月份  全台平均 AQI
 01 月    68.3 ██████████████████████████████████
 02 月    71.0 ███████████████████████████████████
 03 月    66.7 █████████████████████████████████
 04 月    68.5 ██████████████████████████████████
 05 月    58.2 █████████████████████████████
 06 月    49.3 ████████████████████████
 07 月    41.2 ████████████████████
 08 月    46.6 ███████████████████████
 09 月    57.9 ████████████████████████████
 10 月    61.3 ██████████████████████████████
 11 月    64.4 ████████████████████████████████
 12 月    64.7 ████████████████████████████████


> 💡 **看懂結果**：**冬天和春天（11 月～隔年 4 月）最差**，平均接近 70；**夏天（6～8 月）最好**，7 月只有 41。
> 原因：冬天吹東北季風，會把境外污染帶來台灣，而且空氣比較不容易擴散；夏天常下午後雷陣雨，把污染物洗掉。**所以冬天出門，更要看一下 AQI。**

<details><summary>🔄 想看最新資料？點我</summary>

想看最近幾天、全台 80 多個測站的 AQI？把這段貼到新的一格執行（需要網路）：

```python
import pandas as pd
url = ('https://data.moenv.gov.tw/api/v2/aqx_p_434?api_key=846e44e1-8cc5-4893-ad87-c79d2d383706'
       '&format=CSV&limit=1000&sort=monitordate%20desc')
recent = pd.read_csv(url)                                  # 最近幾天、全台所有測站的 AQI
recent.head()
```

> ⚠️ 開放資料的欄位偶爾會改名或調整，如果最新資料跑出錯誤，正好練習看懂錯誤訊息、對照欄位名稱修正程式。

</details>

---

## ✍️ 換你動手：練習區

每一題都有三格：**① 半成品程式**（把 `___` 換成正確的程式）→ **② ✅ 檢查格**（自動告訴你對不對）→ **③ 🔑 參考解答**（折疊起來，先自己試）。

> 看到 `NameError: name '___' is not defined` 不用緊張，那代表還有空格沒填。

In [9]:
# @title 🧰 練習檢查小幫手（執行一次就好，不用看懂裡面的程式）
import math
import numpy as np
import pandas as pd


def _same(ans, expected, ordered=False):
    if isinstance(expected, (pd.DataFrame, pd.Series)):
        if type(ans) is not type(expected):
            return False
        try:
            assert_equal = pd.testing.assert_frame_equal if isinstance(expected, pd.DataFrame) else pd.testing.assert_series_equal
            if not ordered:
                ans, expected = ans.sort_index(), expected.sort_index()
            assert_equal(ans, expected, check_names=False, check_dtype=False,
                         check_index_type=False, check_exact=False, rtol=1e-3, atol=0.06,
                         **({'check_column_type': False} if isinstance(expected, pd.DataFrame) else {}))
            return True
        except (AssertionError, TypeError, ValueError):
            return False
    if isinstance(expected, np.ndarray):
        return np.asarray(ans).shape == expected.shape and np.allclose(np.asarray(ans, dtype=float), expected, atol=0.06)
    if isinstance(expected, (int, float, np.number)) and not isinstance(expected, bool):
        try:
            return math.isclose(float(ans), float(expected), rel_tol=1e-3, abs_tol=0.06)
        except (TypeError, ValueError):
            return False
    if isinstance(expected, (list, tuple, set)):
        return list(ans) == list(expected) if not isinstance(expected, set) else set(ans) == expected
    return ans == expected


def check(name, expected, ordered=False):
    """比對學生的變數 name 和正確答案 expected，印出 ✅ 或 ❌；ordered=True 時連排列順序也要一樣。"""
    if name not in globals():
        print(f'⏳ 還找不到變數 {name}：請先把上一格的 ___ 填好並執行')
        return
    ans = globals()[name]
    if _same(ans, expected, ordered):
        print(f'✅ {name} 答對了！')
        return
    print(f'❌ {name} 還不對，再檢查看看：')
    if type(ans) is not type(expected):
        print(f'   型態不同：你的是 {type(ans).__name__}，應該是 {type(expected).__name__}')
    elif hasattr(expected, 'shape') and getattr(ans, 'shape', None) != expected.shape:
        print(f'   大小不同：你的是 {ans.shape}，應該是 {expected.shape}')
    else:
        print('   型態和大小都對了，是數值' + ('或排列順序' if ordered else '') + '不一樣——檢查一下篩選條件或計算方式')


print('🧰 檢查小幫手準備好了：每題寫完後，執行下方的「✅ 檢查」格')

🧰 檢查小幫手準備好了：每題寫完後，執行下方的「✅ 檢查」格


### ✍️ 練習 1：金門的 AQI，中位數和平均數差多少？

<details><summary>💡 卡住了？點我看提示</summary>

`aqi[:, 欄號]` 取出一整欄；中位數是 `np.median()`。金門這一欄沒有 nan，可以直接用 `.mean()`。

</details>

In [ ]:
# ✏️ 練習 1：把 ___ 換成正確的程式，再按執行
kinmen = aqi[:, sites.index(___)]                   # 取出金門那一欄
kinmen_median = np.___(kinmen)                     # 中位數
kinmen_mean = round(kinmen.mean(), 1)
print('中位數：', kinmen_median, '平均：', kinmen_mean)

In [ ]:
# @title ✅ 執行這格，檢查練習 1
_k = aqi[:, sites.index('金門')]
check('kinmen_median', np.median(_k))
check('kinmen_mean', round(_k.mean(), 1))

<details><summary>🔑 先自己試試看！真的想不出來再點開參考解答</summary>

```python
kinmen = aqi[:, sites.index('金門')]
kinmen_median = np.median(kinmen)
kinmen_mean = round(kinmen.mean(), 1)
print('中位數：', kinmen_median, '平均：', kinmen_mean)
# 平均比中位數高：少數幾天特別嚴重的污染，把平均往上拉
```

</details>

### ✍️ 練習 2：斗六空氣最差的 3 天是哪幾天？

<details><summary>💡 卡住了？點我看提示</summary>

`np.argsort(陣列)` 回傳「由小到大排序後的位置」，取最後 3 個 `[-3:]` 再反轉 `[::-1]` 就是最大的 3 個。

</details>

In [ ]:
# ✏️ 練習 2：把 ___ 換成正確的程式，再按執行
douliu = aqi[:, sites.index('斗六')]
top3 = np.___(douliu)[-3:][::-1]                   # AQI 最高 3 天的位置（由大到小）
for i in top3:
    print(dates[i], douliu[i])

In [ ]:
# @title ✅ 執行這格，檢查練習 2
check('top3', np.argsort(aqi[:, sites.index('斗六')])[-3:][::-1])

<details><summary>🔑 先自己試試看！真的想不出來再點開參考解答</summary>

```python
douliu = aqi[:, sites.index('斗六')]
top3 = np.argsort(douliu)[-3:][::-1]
for i in top3:
    print(dates[i], douliu[i])
```

</details>

### ✍️ 練習 3：紅色警戒！每個測站有幾天 AQI ≥ 150？

<details><summary>💡 卡住了？點我看提示</summary>

AQI 151 以上是「對所有族群不健康」（紅色）。`(aqi >= 150)` 得到 True/False，`.sum(axis=0)` 直著數，每個測站一個數字。

</details>

In [ ]:
# ✏️ 練習 3：把 ___ 換成正確的程式，再按執行
red_days = (aqi >= ___).sum(axis=___)
print(dict(zip(sites, red_days.tolist())))

In [ ]:
# @title ✅ 執行這格，檢查練習 3
check('red_days', (aqi >= 150).sum(axis=0))

<details><summary>🔑 先自己試試看！真的想不出來再點開參考解答</summary>

```python
red_days = (aqi >= 150).sum(axis=0)
print(dict(zip(sites, red_days.tolist())))
```

</details>

---

## 🏆 任務完成！今天你用程式做到了這些事

- ✅ 任務 1：把 CSV 讀成一個 NumPy 矩陣
- ✅ 任務 2：每個測站的年平均——遇到 nan 怎麼辦？
- ✅ 任務 3：全台平均最差的是哪一天？
- ✅ 任務 4：一年之中，有幾天「對敏感族群不健康」？
- ✅ 任務 5：替每一天貼上標籤——像 Excel 的 IF
- ✅ 任務 6：什麼季節空氣最差？

> 🎯 **帶走一句話**：程式只是工具，真正有價值的是你從資料裡找到的那句「所以我們應該……」。
> 把上面每個 💡 **看懂結果** 用自己的話寫成一句建議，就是一份可以拿去報告的分析。